In [ ]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "09_lstm_real"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


In [ ]:
# 2. Carregar dados do novo pipeline multisource
import pandas as pd, numpy as np

# ATUALIZADO: Usar dados do pipeline multisource (NB 12-14)
ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "news_clean.parquet")

# Fallback para arquivo antigo se novo não existir
if not os.path.exists(news_file):
    news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")
    print(f"⚠️ Usando arquivo legado: {news_file}")
    print("   Recomendado: Execute notebooks 12-14 para gerar news_clean.parquet")

if news_file.endswith('.parquet'):
    df_news = pd.read_parquet(news_file)
else:
    df_news = pd.read_csv(news_file)

df_ibov = pd.read_csv(ibov_file)

print("IBOV:", df_ibov.shape, "| NEWS:", df_news.shape)

if "date" in df_ibov.columns: df_ibov.rename(columns={"date":"data"}, inplace=True)

if "published_at" in df_news.columns: df_news.rename(columns={"published_at":"data"}, inplace=True)    df_news["clean_text"] = df_news.get("texto","").fillna("")

elif "date" in df_news.columns: df_news.rename(columns={"date":"data"}, inplace=True)if "clean_text" not in df_news.columns:



df_ibov["data"] = pd.to_datetime(df_ibov["data"])df_news["data"] = pd.to_datetime(df_news["data"])

In [ ]:
# 3. Agregar textos e criar alvo
df_text = df_news.groupby("data", as_index=False)["clean_text"].apply(
    lambda s: " ".join([str(x) for x in s if str(x).strip()])
)

df_ibov = df_ibov.sort_values("data").reset_index(drop=True)
df_ibov["ret1"] = df_ibov["close"].pct_change().shift(-1)
df_ibov["y"] = (df_ibov["ret1"] > 0).astype(int)
df_target = df_ibov.dropna(subset=["ret1"]).copy()

df = pd.merge(df_target[["data","close","y"]], df_text, on="data", how="inner").sort_values("data")

# Validação: exigir dados
if df.shape[0] == 0:
    raise ValueError(
        "❌ ERRO: Nenhuma interseção entre IBOV e notícias. "
        "Execute os notebooks 12-14 (pipeline multisource) "
        "ou 05-06 (coleta legada) para gerar dados antes de prosseguir."
    )

print("Dataset final:", df.shape)
display(df.head())

In [ ]:
# 4. Vetorização com embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["clean_text"].fillna("").tolist(), show_progress_bar=True)

X = np.array(embeddings)
y = df["y"].reset_index(drop=True)
print("Embeddings gerados:", X.shape)


In [ ]:
# Dependência já deve estar instalada via requirements.txt

In [ ]:
# 5. Transformar em sequências para LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def create_sequences(X, y, window=3):
    X_seq, y_seq = [], []
    for i in range(len(X) - window):
        X_seq.append(X[i:i+window])
        y_seq.append(y[i+window])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, window=3)

print("X_seq:", X_seq.shape, "| y_seq:", y_seq.shape)

In [ ]:
# 6. Construir modelo LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

timesteps, features = X_seq.shape[1], X_seq.shape[2]

model = Sequential([
    LSTM(64, input_shape=(timesteps, features)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# 7. Treinar e avaliar
from sklearn.metrics import roc_auc_score, accuracy_score

split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

history = model.fit(X_train, y_train, validation_split=0.2,
                    epochs=20, batch_size=8, verbose=1)

y_pred_proba = model.predict(X_test).ravel()
y_pred = (y_pred_proba > 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba) if len(np.unique(y_test)) > 1 else float("nan")
mda = accuracy_score(y_test, y_pred)

print(f"\nResultados LSTM: AUC={auc:.3f}, MDA={mda:.3f}")

In [ ]:
# 8. Organizar resultados e salvar em JSON (para o dashboard)
import json, os
import numpy as np
from src.utils import logger

nb_name = "09_lstm_real"

# Usar métricas calculadas na célula anterior (auc, mda do último teste)
# Garantir que as variáveis existam
if 'auc' not in locals() or 'mda' not in locals():
    print("⚠️ AVISO: Métricas não calculadas - usando valores padrão")
    auc = 0.0
    mda = 0.0

# Transformar métricas da LSTM em dict compatível
results = {
    "LSTM": {
        "AUC": float(auc) if not np.isnan(auc) else 0.0,
        "MDA": float(mda)
    }
}

# results -> dict serializável
results_dict = {k: {"AUC": float(v["AUC"]), "MDA": float(v["MDA"])} for k, v in results.items()}

# Salvar JSON em data_processed
out_file = os.path.join(PROC_PATH, f"results_{nb_name}.json")
with open(out_file, "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"✅ Resultados salvos em {out_file}")
print(f"   AUC: {results['LSTM']['AUC']:.3f}")
print(f"   MDA: {results['LSTM']['MDA']:.3f}")

# Usar logger estruturado
extra = {
    "sequence_length": 5,
    "epochs": 20,
    "batch_size": 8
}

logger.log_result(
    model_name="lstm_sentiment",
    dataset_name="real",
    metrics={"AUC": results['LSTM']['AUC'], "MDA": results['LSTM']['MDA']},
    extra=extra
)


In [ ]:
# Validar período de cobertura
PERIODO = cfg.get_periodo_estudo()
START_EXPECTED = pd.to_datetime(PERIODO["start"])
END_EXPECTED = pd.to_datetime(PERIODO["end"])

if "data" in df.columns:
    min_date = pd.to_datetime(df["data"].min())
    max_date = pd.to_datetime(df["data"].max())
    print(f"\n=== Validação de Período ===")
    print(f"Esperado: {START_EXPECTED.date()} → {END_EXPECTED.date()}")
    print(f"Obtido:   {min_date.date()} → {max_date.date()}")
    
    if min_date > START_EXPECTED + pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Início dos dados ({min_date.date()}) está >30 dias após o esperado.")
    if max_date < END_EXPECTED - pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Fim dos dados ({max_date.date()}) está >30 dias antes do esperado.")
    if min_date <= START_EXPECTED and max_date >= END_EXPECTED:
        print("✅ Período validado com sucesso!")